# 06 · Reprojection-Based Cloud Filtering  ← **Key visualisation**

**Goal:** clean the COLMAP sparse cloud by reprojecting every 3D point
into every frame and keeping only the points that consistently land on
structure pixels (per the masks from notebook 05). This is the **core
novelty** of the project: it's what recovers thin branches that raw SfM
triangulates noisily.

The single most persuasive image in the entire project is the
raw-vs-filtered side-by-side that this notebook produces — see
`outputs/figures/06_*_filter_comparison.png`.

From scratch: `src/filter_cloud.py::filter_cloud_by_masks`.

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import filter_cloud, segmentation, sfm, viz
    import cv2

    TREE_ID = "tree_oak_01"
    MASK_VARIANT = "sam"   # swap to "classical" for the ablation

    FRAMES_DIR = PROJECT_ROOT / "data" / "frames" / TREE_ID
    MASKS_DIR = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_masks" / MASK_VARIANT
    POSES = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_poses.npz"
    PLY = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_sparse.ply"


if not PLY.exists():
    raise FileNotFoundError(
        f"Sparse cloud PLY not found at {PLY}.\n"
        "Run notebook 04 first."
    )


if not POSES.exists():
    raise FileNotFoundError(
        f"Per-frame projection matrices not found at {POSES}.\n"
        "Run notebook 04 first."
    )


if not MASKS_DIR.exists():
    raise FileNotFoundError(
        f"{MASK_VARIANT} masks not found at {MASKS_DIR}.\n"
        "Run notebook 05 first."
    )

## 1. Load the raw cloud + per-frame masks + projection matrices

In [ ]:
points_3d, _ = sfm.load_ply(PLY)
poses = np.load(POSES, allow_pickle=True)
Ps = list(poses["projections"])
image_names = list(poses["image_names"])

masks = []
for name in image_names:
    stem = Path(name).stem
    mp = MASKS_DIR / (stem + ".png")
    if not mp.exists():
        raise FileNotFoundError(f"Missing mask for frame {stem}: {mp}")
    masks.append(segmentation.load_mask(mp))

print(f"Cloud: {len(points_3d)} pts; Frames: {len(Ps)}; Masks: {len(masks)}")

## 2. Apply the reprojection filter

In [ ]:
kept_pts, kept_mask, diag = filter_cloud.filter_cloud_by_masks(
    points_3d, Ps, masks,
    hit_rate_threshold=0.7,
    min_visible_frames=4,
)
print(f"Survivors: {len(kept_pts)} / {len(points_3d)}  ({100*len(kept_pts)/len(points_3d):.1f}%)")
print(f"Median hit-rate of survivors: {np.median(diag.hit_rate[kept_mask]):.3f}")

## 3. Raw vs. filtered — the headline figure

In [ ]:
fig = viz.plot_cloud_pair(
    points_3d, kept_pts,
    title_raw=f"Raw COLMAP cloud",
    title_filtered=f"After {MASK_VARIANT} mask filter",
    suptitle=f"{TREE_ID} — reprojection filter (hit-rate ≥ 0.7, min 4 frames)",
)
viz.save_fig(fig, f"06_{TREE_ID}_{MASK_VARIANT}_filter_comparison.png")
plt.show()

## 4. Save the filtered cloud for the trunk/skeleton stage

In [ ]:
out = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_filtered_{MASK_VARIANT}.ply"
sfm.save_ply(out, kept_pts)
print(f"Filtered cloud → {out}")